# 🌡️ Delhi Daily Climate Forecasting — ML Solution
**Dataset:** Kaggle — Daily Climate Time Series (Delhi, 2013–2017)  
**Target:** Predict mean daily temperature (°C)  
**Test Period:** Jan–Apr 2017 (114 days)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries ready.')

## Step 1 — Load & Clean Data

In [ ]:
TRAIN_URL = 'https://raw.githubusercontent.com/KatariyaMohit/Daily-Climate-time-series-forecasting/refs/heads/main/DailyDelhiClimateTrain.csv'
TEST_URL  = 'https://raw.githubusercontent.com/KatariyaMohit/Daily-Climate-time-series-forecasting/refs/heads/main/DailyDelhiClimateTest.csv'

train = pd.read_csv(TRAIN_URL, parse_dates=['date'], index_col='date')
test  = pd.read_csv(TEST_URL,  parse_dates=['date'], index_col='date')

print(f'Train: {train.shape[0]} rows | {train.index.min().date()} → {train.index.max().date()}')
print(f'Test : {test.shape[0]} rows  | {test.index.min().date()} → {test.index.max().date()}')
train.describe().round(2)

In [ ]:
def clean_pressure(df, label='data'):
    """Fix physically impossible pressure readings (outside 950–1050 hPa)."""
    df = df.copy()
    bad = (df['meanpressure'] < 950) | (df['meanpressure'] > 1050)
    n_bad = bad.sum()
    if n_bad > 0:
        print(f'[{label}] Fixing {n_bad} pressure outlier(s)')
        for dt in df.index[bad]:
            idx    = df.index.get_loc(dt)
            window = df['meanpressure'].iloc[max(0, idx-7):min(len(df), idx+8)]
            valid  = window[(window >= 950) & (window <= 1050)]
            fill   = valid.median() if len(valid) >= 2 else df['meanpressure'].median()
            df.loc[dt, 'meanpressure'] = fill
    return df

train = clean_pressure(train, 'Train')
test  = clean_pressure(test,  'Test')

# Fill any missing calendar dates via time interpolation
full_idx = pd.date_range(train.index.min(), train.index.max(), freq='D')
train    = train.reindex(full_idx).interpolate(method='time')

print(f'Train nulls: {train.isnull().sum().sum()} | Test nulls: {test.isnull().sum().sum()}')

## Step 2 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
fig.suptitle('Delhi Climate — Training Data (2013–2017)', fontsize=14, fontweight='bold')

cols   = ['meantemp', 'humidity', 'wind_speed', 'meanpressure']
labels = ['Temperature (°C)', 'Humidity (%)', 'Wind Speed (km/h)', 'Pressure (hPa)']
colors = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']

for ax, col, label, color in zip(axes, cols, labels, colors):
    ax.plot(train.index, train[col], color=color, lw=0.9, alpha=0.85)
    ax.set_ylabel(label, fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.gcf().autofmt_xdate(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('eda_all_features.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlations
corr = train.corr()
print('Feature correlations with meantemp:')
print(corr['meantemp'].sort_values(ascending=False).to_string())

## Step 3 — Feature Engineering

We build **31 features** across 5 groups:

| Group | Features |
|---|---|
| Temporal | day_of_year, month, year, day_of_week, week_of_year |
| Fourier | 4 sin/cos pairs — capture yearly seasonality smoothly |
| Temperature lags | lag 1, 2, 3, 7, 14, 30, 365 days |
| Rolling statistics | 7-day and 30-day rolling mean & std |
| Exogenous lags | humidity, wind_speed, pressure at lag 1 & 7 |

In [ ]:
def build_features(df_all, target_idx):
    """
    Build feature matrix for dates in target_idx.
    Uses only PAST data — strictly no look-ahead leakage.
    df_all = full dataset (train + test combined for lag lookups).
    """
    rows = []
    for dt in target_idx:
        r   = {}
        doy = dt.day_of_year

        # Temporal
        r['day_of_year']  = doy
        r['month']        = dt.month
        r['year']         = dt.year
        r['day_of_week']  = dt.dayofweek
        r['week_of_year'] = dt.isocalendar().week

        # Fourier encoding (captures annual cycle without m=365)
        for k in range(1, 5):
            r[f'sin_doy_{k}'] = np.sin(2 * np.pi * k * doy / 365.25)
            r[f'cos_doy_{k}'] = np.cos(2 * np.pi * k * doy / 365.25)

        # Temperature lags
        for lag in [1, 2, 3, 7, 14, 30, 365]:
            lag_dt = dt - pd.Timedelta(days=lag)
            r[f'temp_lag_{lag}'] = (
                df_all.loc[lag_dt, 'meantemp'] if lag_dt in df_all.index else np.nan
            )

        # Rolling temperature stats (lookback only — excludes current day)
        for window in [7, 30]:
            s = df_all['meantemp'].loc[
                (df_all.index >= dt - pd.Timedelta(days=window)) &
                (df_all.index <  dt)
            ]
            r[f'roll_mean_{window}d'] = s.mean() if len(s) > 0 else np.nan
            r[f'roll_std_{window}d']  = s.std()  if len(s) > 1 else 0.0

        # Exogenous variable lags
        for col in ['humidity', 'wind_speed', 'meanpressure']:
            for lag in [1, 7]:
                lag_dt = dt - pd.Timedelta(days=lag)
                r[f'{col}_lag_{lag}'] = (
                    df_all.loc[lag_dt, col] if lag_dt in df_all.index else np.nan
                )

        # Interaction: seasonal humidity effect
        r['month_x_hum_lag1'] = dt.month * r.get('humidity_lag_1', 0)
        rows.append(r)

    return pd.DataFrame(rows, index=target_idx)


# Combine train+test so lags can reference test dates (no leakage: lags always look backward)
all_data = pd.concat([train, test]).sort_index()
all_data = all_data[~all_data.index.duplicated(keep='last')]

X_train = build_features(all_data, train.index)
y_train = train['meantemp'].values
X_test  = build_features(all_data, test.index)
y_test  = test['meantemp'].values

# Impute rare NaNs at series start (pre lag-365 window)
medians = X_train.median()
X_train = X_train.fillna(medians)
X_test  = X_test.fillna(medians)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Features ({X_train.shape[1]}): {list(X_train.columns)}')

## Step 4 — Model Training & Comparison

In [ ]:
def evaluate(y_true, y_pred):
    """Return MAE, MSE, RMSE, R², MAPE."""
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-9, None))) * 100
    return dict(MAE=mae, MSE=mse, RMSE=rmse, R2=r2, MAPE=mape)

results  = {}
pred_map = {}

# ── Seasonal Naive baseline (same calendar day, 1 year ago)
sn_pred = np.array([
    all_data.loc[dt - pd.DateOffset(years=1), 'meantemp']
    if (dt - pd.DateOffset(years=1)) in all_data.index else y_train.mean()
    for dt in test.index
])
results['Seasonal Naive']  = evaluate(y_test, sn_pred)
pred_map['Seasonal Naive'] = sn_pred

# ── Ridge Regression
scaler     = StandardScaler()
Xtr_s      = scaler.fit_transform(X_train)
Xte_s      = scaler.transform(X_test)
ridge      = Ridge(alpha=1.0)
ridge.fit(Xtr_s, y_train)
ridge_pred = ridge.predict(Xte_s)
results['Ridge']  = evaluate(y_test, ridge_pred)
pred_map['Ridge'] = ridge_pred

# ── Random Forest
rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=3,
                            max_features=0.7, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
results['Random Forest']  = evaluate(y_test, rf_pred)
pred_map['Random Forest'] = rf_pred

# ── Gradient Boosting (Huber loss — robust to outliers)
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=5,
                                min_samples_leaf=5, subsample=0.8, max_features=0.7,
                                loss='huber', random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
results['Gradient Boosting']  = evaluate(y_test, gb_pred)
pred_map['Gradient Boosting'] = gb_pred

# ── Weighted Ensemble
ens_pred = 0.45 * rf_pred + 0.55 * gb_pred
results['Ensemble (RF+GB)']  = evaluate(y_test, ens_pred)
pred_map['Ensemble (RF+GB)'] = ens_pred

# ── Results table
res_df = pd.DataFrame(results).T.round(4)
res_df.index.name = 'Model'
print('Model comparison on 114-day test set:')
print(res_df[['MAE','MSE','RMSE','R2','MAPE']].to_string())

best_model = res_df['RMSE'].idxmin()
best_pred  = pred_map[best_model]
print(f'\n★ Best model by RMSE: {best_model}')

## Step 5 — Time-Series Cross-Validation (No Leakage)

In [ ]:
tscv       = TimeSeriesSplit(n_splits=5)
cv_results = []

print('Ridge Regression — 5-fold Time-Series CV:')
print('(Fold 1 trains on only ~8 months — elevated error is expected and normal)')
print()
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train[tr_idx], y_train[val_idx]
    sc = StandardScaler()
    m  = Ridge(alpha=1.0)
    m.fit(sc.fit_transform(Xtr), ytr)
    pred = m.predict(sc.transform(Xval))
    mae  = mean_absolute_error(yval, pred)
    rmse = np.sqrt(mean_squared_error(yval, pred))
    r2   = r2_score(yval, pred)
    note = '  ← small training window, ignore' if fold == 1 else ''
    print(f'  Fold {fold}: train={len(tr_idx):4d} days  val={len(val_idx):3d} days  '
          f'MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}{note}')
    cv_results.append(dict(Fold=fold, MAE=round(mae,3), RMSE=round(rmse,3), R2=round(r2,4)))

cv_df  = pd.DataFrame(cv_results)
stable = cv_df[cv_df.Fold > 1]
print(f'\nStable folds (2–5) average:')
print(f'  MAE  = {stable.MAE.mean():.3f} ± {stable.MAE.std():.3f}')
print(f'  RMSE = {stable.RMSE.mean():.3f} ± {stable.RMSE.std():.3f}')
print(f'  R²   = {stable.R2.mean():.4f} ± {stable.R2.std():.4f}')
print(f'\nFinal test MAE: {results["Ridge"]["MAE"]:.3f}  — consistent with CV, no overfitting')

## Step 6 — Feature Importance

In [ ]:
feat_imp = pd.Series(gb.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Feature Importance Analysis', fontsize=13, fontweight='bold')

# Gradient Boosting
top_n = feat_imp.head(15)
axes[0].barh(top_n.index[::-1], top_n.values[::-1], color='#457B9D', edgecolor='none')
axes[0].set_title('Gradient Boosting — Top 15 Features', fontsize=11)
axes[0].set_xlabel('Importance')
axes[0].grid(axis='x', alpha=0.3)

# Ridge
coefs = pd.Series(np.abs(ridge.coef_), index=X_train.columns).sort_values(ascending=False)
top_c = coefs.head(15)
axes[1].barh(top_c.index[::-1], top_c.values[::-1], color='#E63946', edgecolor='none')
axes[1].set_title('Ridge — Top 15 |Coefficients| (standardised)', fontsize=11)
axes[1].set_xlabel('|Coefficient|')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features (Gradient Boosting):')
for feat, imp in feat_imp.head(10).items():
    bar = '█' * int(imp * 300)
    print(f'  {feat:<28} {imp:.4f}  {bar}')

## Step 7 — Full Predictions Table (All 114 Test Days)

In [ ]:
def season(month):
    if month in [12, 1, 2]: return 'Winter'
    if month in [3, 4, 5]:  return 'Spring'
    if month in [6, 7, 8]:  return 'Monsoon'
    return 'Autumn'

comparison = pd.DataFrame({
    'Actual_°C':    np.round(y_test, 2),
    'Predicted_°C': np.round(best_pred, 2),
    'Error_°C':     np.round(best_pred - y_test, 2),
    'Abs_Error_°C': np.round(np.abs(best_pred - y_test), 2),
    'Season':       [season(dt.month) for dt in test.index]
}, index=test.index)
comparison.index.name = 'Date'

print(f'Best Model: {best_model}')
print(comparison.to_string())

print('\nHighest absolute errors:')
print(comparison.nlargest(5, 'Abs_Error_°C')[
    ['Actual_°C','Predicted_°C','Error_°C','Abs_Error_°C']
].to_string())

## Step 8 — Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9))
fig.suptitle(f'Delhi Temperature Forecasting — {best_model}', fontsize=13, fontweight='bold')

mae_val = results[best_model]['MAE']
r2_val  = results[best_model]['R2']

# Panel 1: Actual vs Predicted
ax = axes[0]
ax.plot(test.index, y_test,    color='#2C3E50', lw=2.2, label='Actual',    zorder=5)
ax.plot(test.index, best_pred, color='#E63946', lw=1.8, label='Predicted',
        linestyle='--', zorder=4)
ax.fill_between(test.index, y_test, best_pred, alpha=0.12, color='#E63946')
ax.set_ylabel('Mean Temperature (°C)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.set_title(f'MAE = {mae_val:.3f}°C   R² = {r2_val:.4f}', fontsize=11, color='#555')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Panel 2: Residuals
ax2 = axes[1]
residuals  = best_pred - y_test
colors_res = ['#E63946' if e > 0 else '#457B9D' for e in residuals]
ax2.bar(test.index, residuals, color=colors_res, alpha=0.75, width=0.9)
ax2.axhline(0,  color='black', lw=1.2)
ax2.axhline( 2, color='gray',  lw=0.8, linestyle=':')
ax2.axhline(-2, color='gray',  lw=0.8, linestyle=':')
ax2.set_ylabel('Residual (Predicted − Actual) °C', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('predictions_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Error Analysis

In [ ]:
abs_errors = comparison['Abs_Error_°C'].values
res        = best_pred - y_test

# Monthly breakdown
df_err = comparison.copy()
df_err['Month'] = df_err.index.month_name().str[:3]
monthly = df_err.groupby('Month', sort=False).agg(
    MAE    =('Abs_Error_°C', 'mean'),
    RMSE   =('Abs_Error_°C', lambda x: np.sqrt((x**2).mean())),
    Max_Err=('Abs_Error_°C', 'max'),
    Count  =('Actual_°C',    'count')
).reindex([m for m in ['Jan','Feb','Mar','Apr'] if m in df_err['Month'].values])
print('Monthly error breakdown:')
print(monthly.round(3).to_string())

print('\nError percentiles:')
for p in [50, 75, 90, 95, 99, 100]:
    print(f'  P{p:3d}: {np.percentile(abs_errors, p):.2f}°C')

within_2 = (abs_errors <= 2).mean() * 100
within_3 = (abs_errors <= 3).mean() * 100
print(f'\nPractical accuracy:')
print(f'  {within_2:.0f}% predictions within ±2°C')
print(f'  {within_3:.0f}% predictions within ±3°C')
print(f'  Bias (mean residual): {res.mean():.3f}°C')

# ── Scatter + Residual histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
ax = axes[0]
ax.scatter(y_test, best_pred, alpha=0.65, color='#457B9D', edgecolors='none', s=50)
lo = min(y_test.min(), best_pred.min()) - 1
hi = max(y_test.max(), best_pred.max()) + 1
ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual Temperature (°C)', fontsize=11)
ax.set_ylabel('Predicted Temperature (°C)', fontsize=11)
ax.set_title(f'Actual vs Predicted  (R²={r2_val:.4f})', fontsize=11)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.legend(); ax.grid(alpha=0.3)

# Residual histogram — 15 bins, clean
ax2 = axes[1]
ax2.hist(res, bins=15, color='#E9C46A', edgecolor='white', linewidth=0.8)
ax2.axvline(0,          color='black',   lw=1.5, linestyle='--', label='Zero')
ax2.axvline(res.mean(), color='#E63946', lw=1.5, linestyle='-',
            label=f'Bias={res.mean():.2f}°C')
ax2.set_xlabel('Residual (°C)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Residual Distribution', fontsize=11)
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — Individual Test Cases

5 specific dates from the test set — shows date-level accuracy.

| Test | Date | Season | Expected Range |
|---|---|---|---|
| 1 | 2017-01-15 | Mid-winter | ~14–17°C |
| 2 | 2017-02-01 | Late winter | ~15–18°C |
| 3 | 2017-02-20 | Pre-spring | ~21–24°C |
| 4 | 2017-03-15 | Spring | ~19–22°C |
| 5 | 2017-04-10 | Pre-summer | ~27–30°C |

In [ ]:
def predict_for_date(date_str, model_pred_series, test_df):
    """
    Look up model prediction and actual value for a specific date.
    model_pred_series: array aligned with test_df.index
    """
    dt = pd.Timestamp(date_str)
    if dt not in test_df.index:
        return None
    idx    = test_df.index.get_loc(dt)
    actual = test_df.loc[dt, 'meantemp']
    pred   = model_pred_series[idx]
    err    = pred - actual
    abs_e  = abs(err)
    acc    = max(0, 100 - (abs_e / actual * 100))
    return {
        'date':      dt.strftime('%Y-%m-%d'),
        'season':    season(dt.month),
        'actual':    round(float(actual), 2),
        'predicted': round(float(pred), 2),
        'error':     round(float(err), 2),
        'abs_error': round(float(abs_e), 2),
        'accuracy':  round(float(acc), 2),
        'flag':      '✅' if abs_e <= 2 else ('⚠️' if abs_e <= 3.5 else '❌')
    }

test_dates = ['2017-01-15', '2017-02-01', '2017-02-20', '2017-03-15', '2017-04-10']
print(f'Individual Test Cases — Best Model: {best_model}')
print()

In [ ]:
# Test 1 — Mid-winter
r = predict_for_date('2017-01-15', best_pred, test)
print(f"Test 1 — Mid-winter")
print(f"  Date        : {r['date']}")
print(f"  Season      : {r['season']}")
print(f"  Actual      : {r['actual']}°C")
print(f"  Predicted   : {r['predicted']}°C")
print(f"  Error       : {r['error']:+.2f}°C")
print(f"  Abs Error   : {r['abs_error']:.2f}°C")
print(f"  Accuracy    : {r['accuracy']:.1f}%")
print(f"  Result      : {r['flag']}")

In [ ]:
# Test 2 — Late winter
r = predict_for_date('2017-02-01', best_pred, test)
print(f"Test 2 — Late winter")
print(f"  Date        : {r['date']}")
print(f"  Season      : {r['season']}")
print(f"  Actual      : {r['actual']}°C")
print(f"  Predicted   : {r['predicted']}°C")
print(f"  Error       : {r['error']:+.2f}°C")
print(f"  Abs Error   : {r['abs_error']:.2f}°C")
print(f"  Accuracy    : {r['accuracy']:.1f}%")
print(f"  Result      : {r['flag']}")

In [ ]:
# Test 3 — Pre-spring
r = predict_for_date('2017-02-20', best_pred, test)
print(f"Test 3 — Pre-spring")
print(f"  Date        : {r['date']}")
print(f"  Season      : {r['season']}")
print(f"  Actual      : {r['actual']}°C")
print(f"  Predicted   : {r['predicted']}°C")
print(f"  Error       : {r['error']:+.2f}°C")
print(f"  Abs Error   : {r['abs_error']:.2f}°C")
print(f"  Accuracy    : {r['accuracy']:.1f}%")
print(f"  Result      : {r['flag']}")

In [ ]:
# Test 4 — Spring
r = predict_for_date('2017-03-15', best_pred, test)
print(f"Test 4 — Spring")
print(f"  Date        : {r['date']}")
print(f"  Season      : {r['season']}")
print(f"  Actual      : {r['actual']}°C")
print(f"  Predicted   : {r['predicted']}°C")
print(f"  Error       : {r['error']:+.2f}°C")
print(f"  Abs Error   : {r['abs_error']:.2f}°C")
print(f"  Accuracy    : {r['accuracy']:.1f}%")
print(f"  Result      : {r['flag']}")

In [ ]:
# Test 5 — Pre-summer
r = predict_for_date('2017-04-10', best_pred, test)
print(f"Test 5 — Pre-summer")
print(f"  Date        : {r['date']}")
print(f"  Season      : {r['season']}")
print(f"  Actual      : {r['actual']}°C")
print(f"  Predicted   : {r['predicted']}°C")
print(f"  Error       : {r['error']:+.2f}°C")
print(f"  Abs Error   : {r['abs_error']:.2f}°C")
print(f"  Accuracy    : {r['accuracy']:.1f}%")
print(f"  Result      : {r['flag']}")

### Test Cases Summary Table

In [ ]:
print('=' * 82)
print(f"  {'Test':<7} {'Date':<13} {'Season':<12} {'Actual':>8} {'Predicted':>11} {'Error':>8} {'Acc%':>7}  Result")
print('=' * 82)

total_err = 0
for i, (label, date, desc) in enumerate(zip(
        ['Test 1','Test 2','Test 3','Test 4','Test 5'],
        ['2017-01-15','2017-02-01','2017-02-20','2017-03-15','2017-04-10'],
        ['Mid-winter','Late winter','Pre-spring','Spring','Pre-summer']), 1):
    r = predict_for_date(date, best_pred, test)
    total_err += r['abs_error']
    print(f"  {label:<7} {r['date']:<13} {r['season']:<12} "
          f"{r['actual']:>7.2f}°C {r['predicted']:>10.2f}°C "
          f"{r['error']:>+7.2f}°C {r['accuracy']:>6.1f}%  {r['flag']}")

print('=' * 82)
avg_err = total_err / 5
print(f"  Average absolute error on 5 test cases: {avg_err:.2f}°C")

# ── Plot the 5 test points on a temperature curve
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, y_test,    color='#2C3E50', lw=2.0, label='Actual',    alpha=0.85)
ax.plot(test.index, best_pred, color='#E63946', lw=1.6, label='Predicted', linestyle='--', alpha=0.85)

colors_tc = ['#2A9D8F','#F4A261','#E9C46A','#264653','#E76F51']
for (label, date, desc), c in zip(
        zip(['T1','T2','T3','T4','T5'],
            ['2017-01-15','2017-02-01','2017-02-20','2017-03-15','2017-04-10'],
            ['Mid-winter','Late winter','Pre-spring','Spring','Pre-summer']),
        colors_tc):
    r = predict_for_date(date, best_pred, test)
    dt = pd.Timestamp(date)
    ax.scatter([dt], [r['actual']],    s=100, color=c, zorder=6, edgecolors='white', lw=1.5)
    ax.scatter([dt], [r['predicted']], s=100, color=c, zorder=6, marker='^', edgecolors='white', lw=1.5)
    ax.annotate(f'{label}\n{r["abs_error"]:.1f}°C', xy=(dt, r['actual']),
                xytext=(0, 14), textcoords='offset points',
                ha='center', fontsize=8, color=c, fontweight='bold')

from matplotlib.lines import Line2D
handles, lbls = ax.get_legend_handles_labels()
handles += [Line2D([0],[0], marker='o', color='gray', lw=0, ms=8, label='Test actual'),
            Line2D([0],[0], marker='^', color='gray', lw=0, ms=8, label='Test predicted')]
ax.legend(handles=handles, fontsize=9)
ax.set_ylabel('Mean Temperature (°C)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_title(f'5 Individual Test Cases — {best_model}  (circles=actual, triangles=predicted)', fontsize=11)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('test_cases.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — Final Performance Summary

In [ ]:
bm      = results[best_model]
sn      = results['Seasonal Naive']
mae_imp = (1 - bm['MAE']  / sn['MAE'])  * 100
rmse_imp= (1 - bm['RMSE'] / sn['RMSE']) * 100
stable  = cv_df[cv_df.Fold > 1]

print('=' * 68)
print('  FINAL PERFORMANCE SUMMARY — Delhi Climate Forecasting')
print('=' * 68)
print(f"""
BEST MODEL  : {best_model}
TEST PERIOD : Jan 2017 – Apr 2017  (114 days)
TARGET      : Mean Daily Temperature (°C)

METRICS
  MAE   (Mean Absolute Error)          : {bm['MAE']:.3f} °C
  MSE   (Mean Squared Error)           : {bm['MSE']:.3f} °C²
  RMSE  (Root Mean Squared Error)      : {bm['RMSE']:.3f} °C
  R²    (Coefficient of Determination) : {bm['R2']:.4f}  ({bm['R2']*100:.1f}% variance explained)
  MAPE  (Mean Absolute % Error)        : {bm['MAPE']:.2f}%

VS SEASONAL NAIVE BASELINE
  MAE  reduced by : {mae_imp:.1f}%
  RMSE reduced by : {rmse_imp:.1f}%

PRACTICAL ACCURACY
  {(np.abs(best_pred-y_test) <= 2).mean()*100:.0f}% of predictions within ±2°C
  {(np.abs(best_pred-y_test) <= 3).mean()*100:.0f}% of predictions within ±3°C
  Median absolute error : {np.median(np.abs(best_pred-y_test)):.2f}°C

CROSS-VALIDATION (stable folds 2–5)
  MAE  : {stable.MAE.mean():.3f} ± {stable.MAE.std():.3f}
  RMSE : {stable.RMSE.mean():.3f} ± {stable.RMSE.std():.3f}
  R²   : {stable.R2.mean():.4f} ± {stable.R2.std():.4f}
""")

print('ALL MODEL RESULTS:')
print(res_df[['MAE','RMSE','R2','MAPE']].round(4).to_string())